# 07 — Storytelling Executivo

Este notebook consolida as respostas às perguntas de negócio, gera `reports/business_questions_report.md` e injeta os dados no dashboard standalone (`dashboard/index.html`).

In [1]:
from __future__ import annotations

from pathlib import Path
import math
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_DIR = Path.cwd().resolve()
while PROJECT_DIR.name != 'amazon-product-intelligence' and PROJECT_DIR.parent != PROJECT_DIR:
    PROJECT_DIR = PROJECT_DIR.parent

PROCESSED_DIR = PROJECT_DIR / 'data' / 'processed'
REPORTS_DIR = PROJECT_DIR / 'reports'
FIGURES_DIR = REPORTS_DIR / 'figures'
DASHBOARD_DIR = PROJECT_DIR / 'dashboard'
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
DASHBOARD_DIR.mkdir(parents=True, exist_ok=True)

import sys
sys.path.insert(0, str(PROJECT_DIR / 'notebooks'))
from src.dashboard import write_dashboard


C:\Users\flavi\Anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


## Carregar bases geradas

In [2]:
df_products = pd.read_csv(PROCESSED_DIR / 'base_produtos_com_clusters.csv')
df_reviews = pd.read_csv(PROCESSED_DIR / 'base_com_sentimento.csv') if (PROCESSED_DIR / 'base_com_sentimento.csv').exists() else pd.read_csv(PROCESSED_DIR / 'base_processada.csv')
(df_products.shape, df_reviews.shape)


((1351, 22), (1465, 30))

## KPIs para dashboard

In [3]:
total_products = int(len(df_products))
avg_rating = float(pd.to_numeric(df_products['rating_clean'], errors='coerce').mean())
avg_discount = float(pd.to_numeric(df_products['discount_pct_clean'], errors='coerce').mean())
avg_savings = float(pd.to_numeric(df_products['economia_absoluta'], errors='coerce').mean())
total_categories = int(df_products['main_category'].nunique(dropna=True))
total_reviews = int(pd.to_numeric(df_products['rating_count_clean'], errors='coerce').fillna(0).sum())
top_category = (df_products['main_category'].value_counts().idxmax() if df_products['main_category'].notna().any() else 'Unknown')
best_psi_row = df_products.sort_values('PSI', ascending=False).iloc[0]
best_psi_product = str(best_psi_row['product_name'])
best_psi_value = float(best_psi_row['PSI'])
max_discount = float(pd.to_numeric(df_products['discount_pct_clean'], errors='coerce').max())
n_clusters = int(df_products['cluster'].nunique())

kpis = {
    'total_products': total_products,
    'avg_rating': round(avg_rating, 2),
    'avg_discount': round(avg_discount, 2),
    'avg_savings': round(avg_savings, 2),
    'total_categories': total_categories,
    'total_reviews': total_reviews,
    'top_category': top_category,
    'best_psi_product': best_psi_product,
    'best_psi_value': round(best_psi_value, 2),
    'max_discount': round(max_discount, 2),
    'n_clusters': n_clusters,
}
kpis


{'total_products': 1351,
 'avg_rating': 4.09,
 'avg_discount': 46.69,
 'avg_savings': 2386.37,
 'total_categories': 9,
 'total_reviews': 23802423,
 'top_category': 'Electronics',
 'best_psi_product': 'Amazon Basics High-Speed HDMI Cable, 6 Feet (2-Pack),Black',
 'best_psi_value': 87.74,
 'max_discount': 94.0,
 'n_clusters': 5}

## Responder perguntas de negócio (gerar Markdown)

In [4]:
def fmt_int(n: int | float) -> str:
    return f"{int(n):,}".replace(',', '.')

def fmt_float(x: float, digits: int = 2) -> str:
    if x is None or (isinstance(x, float) and math.isnan(x)):
        return 'n/a'
    return f"{float(x):.{digits}f}"

def iqr_outlier_share(series: pd.Series) -> float:
    s = pd.to_numeric(series, errors='coerce').dropna()
    if s.empty:
        return 0.0
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return float(((s < lo) | (s > hi)).mean() * 100)

answers: list[dict[str, str]] = []

cat_counts = df_products['main_category'].fillna('Unknown').value_counts()
answers.append({
    'q': 'Qual é o volume e a distribuição de produtos por categoria principal e subcategoria?',
    'a': f"Total de produtos: {fmt_int(total_products)}. Top categoria: {cat_counts.index[0]} ({fmt_int(cat_counts.iloc[0])}).",
    'insight': 'A concentração em poucas categorias sugere oportunidades claras de segmentação e otimização por vertical.',
    'rec': 'Priorizar ações nas maiores categorias e replicar a abordagem nas menores com maior crescimento.'
})

critical_cols = ['product_id','product_name','main_category','discounted_price_clean','actual_price_clean','discount_pct_clean','rating_clean','rating_count_clean']
critical_missing_pct = float(df_products[critical_cols].isna().mean().mean() * 100)
no_rating_pct = float(df_products['rating_clean'].isna().mean() * 100)
no_reviews_pct = float(df_products['rating_count_clean'].isna().mean() * 100)
answers.append({
    'q': 'Qual é a taxa de completude dos dados? Existem produtos sem avaliação ou com dados críticos ausentes?',
    'a': f"Missing médio em campos críticos: {fmt_float(critical_missing_pct,2)}%. Produtos sem rating: {fmt_float(no_rating_pct,2)}%. Produtos sem rating_count: {fmt_float(no_reviews_pct,2)}%.",
    'insight': 'Campos de preço e rating são a base para PSI e clusterização; faltas reduzem comparabilidade.',
    'rec': 'Tratar nulos com imputação no pipeline e isolar itens com dados insuficientes para evitar distorções.'
})

out_share = iqr_outlier_share(df_products['discounted_price_clean'])
answers.append({
    'q': 'Qual é a distribuição de preços por categoria? Existem outliers extremos que distorcem a análise?',
    'a': f"Outliers (IQR) em preço descontado: {fmt_float(out_share,2)}% dos produtos.",
    'insight': 'Outliers tendem a inflar médias; mediana e quartis são mais robustos para comparação por categoria.',
    'rec': 'Usar mediana/IQR em comparações e aplicar filtros ao analisar extremos.'
})

disc_by_cat = df_products.groupby('main_category', dropna=False)['discount_pct_clean'].mean().sort_values(ascending=False)
disc_corr = df_products[['discount_pct_clean','rating_clean']].corr(method='spearman', numeric_only=True).iloc[0,1]
answers.append({
    'q': 'Quais categorias oferecem os maiores descontos médios? Os descontos estão correlacionados com ratings mais altos ou mais baixos?',
    'a': f"Maior desconto médio: {disc_by_cat.index[0]} ({fmt_float(disc_by_cat.iloc[0],2)}%). Correlação Spearman desconto×rating: {fmt_float(disc_corr,3)}.",
    'insight': 'Desconto alto não implica necessariamente melhor percepção; a correlação orienta estratégias por categoria.',
    'rec': 'Ajustar campanhas por categoria considerando o efeito observado em rating.'
})

rating_by_band = df_products.groupby('faixa_preco', dropna=False)['rating_clean'].mean().sort_values(ascending=False)
best_band = str(rating_by_band.index[0])
answers.append({
    'q': 'Existe uma faixa de preço ideal onde os produtos concentram as melhores avaliações?',
    'a': f"Maior rating médio por faixa_preco: {best_band} (rating médio {fmt_float(rating_by_band.iloc[0],2)}).",
    'insight': 'O melhor rating médio por faixa pode indicar um sweet spot de valor percebido.',
    'rec': 'Otimizar sortimento e promoções para maximizar presença na faixa com melhor rating médio.'
})

value_metric = df_products.assign(value_score=df_products['discount_pct_clean'] * df_products['rating_clean']).groupby('main_category')['value_score'].mean().sort_values(ascending=False)
answers.append({
    'q': 'Qual é o desconto médio praticado por categoria e qual categoria entrega mais valor real ao consumidor (desconto + rating alto)?',
    'a': f"Categoria com maior score (desconto×rating): {value_metric.index[0]} ({fmt_float(value_metric.iloc[0],2)}).",
    'insight': 'Combinar desconto e rating evita promover itens baratos mas mal avaliados.',
    'rec': 'Usar o score de valor para priorizar vitrines e campanhas.'
})

hi_disc = df_products[df_products['discount_pct_clean'] > 50]
lo_disc = df_products[df_products['discount_pct_clean'] <= 50]
delta_rating = float(pd.to_numeric(hi_disc['rating_clean'], errors='coerce').mean() - pd.to_numeric(lo_disc['rating_clean'], errors='coerce').mean())
answers.append({
    'q': 'Produtos com desconto acima de 50% têm performance de avaliação diferente dos demais?',
    'a': f"Diferença de rating médio (desconto>50% − <=50%): {fmt_float(delta_rating,3)}.",
    'insight': 'Diferença pequena sugere que desconto não é o único driver de satisfação.',
    'rec': 'Cruzar com volume de reviews e métricas operacionais (quando disponíveis) para validar promoções agressivas.'
})

leaders = df_products.dropna(subset=['rating_clean','rating_count_clean']).copy()
leaders = leaders[(leaders['rating_count_clean'] >= leaders['rating_count_clean'].quantile(0.9))]
leaders = leaders.sort_values(['rating_clean','rating_count_clean'], ascending=False).head(10)
answers.append({
    'q': 'Quais são os produtos mais bem avaliados com alto volume de reviews (os verdadeiros líderes)?',
    'a': f"Top líder: {leaders.iloc[0]['product_name']} (rating {fmt_float(leaders.iloc[0]['rating_clean'],2)}, reviews {fmt_int(leaders.iloc[0]['rating_count_clean'])}).",
    'insight': 'Líderes combinam reputação e escala, sendo âncoras de confiança para a categoria.',
    'rec': 'Proteger disponibilidade e ranqueamento desses itens e usar como benchmark.'
})

corr_reviews_rating = df_products[['rating_count_clean','rating_clean']].corr(method='spearman', numeric_only=True).iloc[0,1]
answers.append({
    'q': 'Existe correlação entre o número de avaliações e o rating médio?',
    'a': f"Correlação Spearman rating_count×rating: {fmt_float(corr_reviews_rating,3)}.",
    'insight': 'Popularidade não garante satisfação; a relação ajuda a guiar estratégia por categoria.',
    'rec': 'Separar líderes, hidden gems e problemas populares para ações distintas.'
})

hidden = df_products.dropna(subset=['rating_clean','rating_count_clean']).copy()
hidden = hidden[(hidden['rating_clean'] >= hidden['rating_clean'].quantile(0.9)) & (hidden['rating_count_clean'] <= hidden['rating_count_clean'].quantile(0.1))]
hidden = hidden.sort_values('rating_clean', ascending=False).head(10)
answers.append({
    'q': 'Quais produtos têm alto rating mas pouquíssimas avaliações (potenciais hidden gems)?',
    'a': f"Exemplo: {hidden.iloc[0]['product_name']} (rating {fmt_float(hidden.iloc[0]['rating_clean'],2)}, reviews {fmt_int(hidden.iloc[0]['rating_count_clean'])})." if len(hidden) else 'Não houve produtos suficientes na interseção (p90 rating e p10 reviews) para listar com confiança.',
    'insight': 'Hidden gems tendem a ter boa qualidade mas baixa descoberta.',
    'rec': 'Testar boosts de visibilidade para validar potencial de escala.'
})

problem = df_products.dropna(subset=['rating_clean','rating_count_clean']).copy()
problem = problem[(problem['rating_clean'] <= problem['rating_clean'].quantile(0.2)) & (problem['rating_count_clean'] >= problem['rating_count_clean'].quantile(0.9))]
problem = problem.sort_values('rating_count_clean', ascending=False).head(10)
answers.append({
    'q': 'Quais produtos têm muitas avaliações mas rating baixo (produtos problemáticos populares)?',
    'a': f"Exemplo: {problem.iloc[0]['product_name']} (rating {fmt_float(problem.iloc[0]['rating_clean'],2)}, reviews {fmt_int(problem.iloc[0]['rating_count_clean'])})." if len(problem) else 'Não houve produtos suficientes na interseção (p20 rating e p90 reviews) para listar com confiança.',
    'insight': 'Itens com alto volume e baixa nota são riscos reputacionais.',
    'rec': 'Auditar causas em reviews e priorizar correções.'
})

df_rank = df_products.sort_values('PSI', ascending=False).reset_index(drop=True)
answers.append({
    'q': 'Qual é o ranking de produtos usando o Product Score Index combinando rating, volume de reviews e desconto?',
    'a': f"#1 por PSI: {df_rank.iloc[0]['product_name']} (PSI {fmt_float(df_rank.iloc[0]['PSI'],2)}).",
    'insight': 'PSI reduz viés de avaliar apenas por rating ou apenas por desconto.',
    'rec': 'Usar PSI como critério de curadoria e comparação entre subcategorias.'
})

top10_cat = (
    df_rank.sort_values(['main_category','PSI'], ascending=[True, False])
    .groupby('main_category', as_index=False)
    .head(10)
)
answers.append({
    'q': 'Quais são os Top 10 produtos por PSI em cada categoria principal?',
    'a': f"Top 10 por categoria exportado em reports/psi_top10_by_category.csv (linhas: {fmt_int(len(top10_cat))}).",
    'insight': 'Ranking por categoria evita comparar produtos incomparáveis.',
    'rec': 'Usar Top 10 por categoria como shortlist para campanhas.'
})

answers.append({
    'q': 'Como os clusters de produtos se distribuem no espaço PSI vs. preço?',
    'a': f"Distribuição no dashboard: PSI vs preço colorido por cluster (clusters: {fmt_int(n_clusters)}).",
    'insight': 'A relação PSI×preço por cluster mostra onde existe alto valor percebido em diferentes faixas.',
    'rec': 'Atuar por cluster com estratégias específicas de preço e promo.'
})

answers.append({
    'q': 'Quantos clusters de produtos existem naturalmente na base (usar Elbow + Silhouette)?',
    'a': f"Clusters usados no modelo final: {fmt_int(n_clusters)} (ver notebook 04 para Elbow + Silhouette).",
    'insight': 'O número de clusters sintetiza padrões de mercado em grupos acionáveis.',
    'rec': 'Reavaliar k periodicamente ao expandir a base.'
})

answers.append({
    'q': 'Quais são os perfis de cada cluster? (ex: premium bem avaliado, barato com alto desconto, popular problemático)',
    'a': 'Perfis e métricas médias por cluster disponíveis no dashboard (cards e tabela).',
    'insight': 'Clusters transformam variáveis contínuas em segmentos com narrativa e estratégia.',
    'rec': 'Atribuir uma estratégia por cluster (pricing, promo, sortimento) e medir impacto.'
})

cluster_profile = (
    df_products.groupby(['cluster','cluster_name'], dropna=False)[['discounted_price_clean','discount_pct_clean','rating_clean','rating_count_clean','PSI']]
    .mean(numeric_only=True)
    .reset_index()
)
cluster_opportunity = cluster_profile.assign(opportunity_score=cluster_profile['rating_clean'] * cluster_profile['discount_pct_clean']).sort_values('opportunity_score', ascending=False)
best_cluster = cluster_opportunity.iloc[0]
answers.append({
    'q': 'Qual cluster representa a maior oportunidade de negócio para a Amazon?',
    'a': f"Cluster com maior score (rating×desconto): {int(best_cluster['cluster'])} — {best_cluster['cluster_name']} (score {fmt_float(best_cluster['opportunity_score'],2)}).",
    'insight': 'A oportunidade combina qualidade percebida e atratividade de preço.',
    'rec': 'Focar visibilidade/estoque nesse cluster e validar uplift em conversão.'
})

if 'sentimento_label' in df_reviews.columns:
    sentiment_share = (
        df_reviews.groupby('main_category', dropna=False)['sentimento_label']
        .value_counts(normalize=True)
        .rename('share')
        .reset_index()
    )
    pred = sentiment_share.sort_values(['main_category','share'], ascending=[True, False]).groupby('main_category').head(1)
    top_sent_row = pred.sort_values('share', ascending=False).iloc[0]
    answers.append({
        'q': 'Qual é o sentimento predominante nos títulos e conteúdos dos reviews por categoria?',
        'a': f"Exemplo: em {top_sent_row['main_category']}, sentimento predominante é {top_sent_row['sentimento_label']} (share {fmt_float(top_sent_row['share']*100,2)}%).",
        'insight': 'Sentimento por categoria complementa rating e pode antecipar problemas de experiência.',
        'rec': 'Monitorar categorias com maior share negativo e cruzar com produtos populares.'
    })
else:
    answers.append({
        'q': 'Qual é o sentimento predominante nos títulos e conteúdos dos reviews por categoria?',
        'a': 'Sentimento não disponível nesta execução (executar notebook 06).',
        'insight': 'Sem análise textual, a visão fica restrita ao rating numérico.',
        'rec': 'Executar análise de sentimento e integrar ao dashboard.'
    })

if 'sentiment_score' in df_reviews.columns:
    r = pd.to_numeric(df_reviews.get('rating_clean', df_reviews.get('rating')), errors='coerce')
    s = pd.to_numeric(df_reviews.get('sentiment_score', np.nan), errors='coerce')
    divergence = float(((r >= 4.0) & (s <= -0.05)).mean() * 100)
    answers.append({
        'q': 'Existe divergência entre o sentimento textual do review e o rating numérico dado?',
        'a': f"Share de reviews com rating>=4 e sentimento negativo: {fmt_float(divergence,2)}%.",
        'insight': 'Divergência sugere que o texto traz nuances além do rating.',
        'rec': 'Usar sentimento como sinal adicional para triagem de problemas.'
    })
else:
    answers.append({
        'q': 'Existe divergência entre o sentimento textual do review e o rating numérico dado?',
        'a': 'Sentimento não disponível nesta execução (executar notebook 06).',
        'insight': 'Sem sentimento, não há como quantificar divergência.',
        'rec': 'Executar notebook 06 e reprocessar.'
    })

def top_words(text_series: pd.Series, n: int = 12) -> list[tuple[str,int]]:
    stop = {
        'the','a','an','and','or','to','of','for','in','on','with','is','it','this','that','was','were','are',
        'i','you','we','they','he','she','my','your','very','not','but','as','at','be','have','has','had',
        'product','amazon','buy','good','bad'
    }
    tokens: list[str] = []
    for t in text_series.dropna().astype(str).tolist():
        t = t.lower()
        t = re.sub(r'[^a-z\s]', ' ', t)
        parts = [p for p in t.split() if len(p) >= 3 and p not in stop]
        tokens.extend(parts)
    if not tokens:
        return []
    s = pd.Series(tokens).value_counts().head(n)
    return list(zip(s.index.tolist(), s.values.tolist()))

if 'sentimento_label' in df_reviews.columns:
    pos_text = (df_reviews.loc[df_reviews['sentimento_label']=='positivo','review_title'].fillna('') + ' ' + df_reviews.loc[df_reviews['sentimento_label']=='positivo','review_content'].fillna('')).str.strip()
    neg_text = (df_reviews.loc[df_reviews['sentimento_label']=='negativo','review_title'].fillna('') + ' ' + df_reviews.loc[df_reviews['sentimento_label']=='negativo','review_content'].fillna('')).str.strip()
    pos_words = top_words(pos_text)
    neg_words = top_words(neg_text)
    answers.append({
        'q': 'Quais palavras mais frequentes aparecem em reviews positivos vs. negativos?',
        'a': f"Top positivos: {pos_words}. Top negativos: {neg_words}.",
        'insight': 'As palavras destacam drivers de satisfação e dor.',
        'rec': 'Usar essas palavras como vocabulário de monitoramento.'
    })
else:
    answers.append({
        'q': 'Quais palavras mais frequentes aparecem em reviews positivos vs. negativos?',
        'a': 'Sentimento não disponível nesta execução (executar notebook 06).',
        'insight': 'Sem sentimento, não há separação positivo/negativo.',
        'rec': 'Executar notebook 06 e reprocessar.'
    })

len(answers)


20

## Gerar o relatório completo (20 perguntas)

In [5]:
expected_q = [
  'Qual é o volume e a distribuição de produtos por categoria principal e subcategoria?',
  'Qual é a taxa de completude dos dados? Existem produtos sem avaliação ou com dados críticos ausentes?',
  'Qual é a distribuição de preços por categoria? Existem outliers extremos que distorcem a análise?',
  'Quais categorias oferecem os maiores descontos médios? Os descontos estão correlacionados com ratings mais altos ou mais baixos?',
  'Existe uma faixa de preço ideal onde os produtos concentram as melhores avaliações?',
  'Qual é o desconto médio praticado por categoria e qual categoria entrega mais valor real ao consumidor (desconto + rating alto)?',
  'Produtos com desconto acima de 50% têm performance de avaliação diferente dos demais?',
  'Quais são os produtos mais bem avaliados com alto volume de reviews (os verdadeiros líderes)?',
  'Existe correlação entre o número de avaliações e o rating médio?',
  'Quais produtos têm alto rating mas pouquíssimas avaliações (potenciais hidden gems)?',
  'Quais produtos têm muitas avaliações mas rating baixo (produtos problemáticos populares)?',
  'Qual é o ranking de produtos usando o Product Score Index combinando rating, volume de reviews e desconto?',
  'Quais são os Top 10 produtos por PSI em cada categoria principal?',
  'Como os clusters de produtos se distribuem no espaço PSI vs. preço?',
  'Quantos clusters de produtos existem naturalmente na base (usar Elbow + Silhouette)?',
  'Quais são os perfis de cada cluster? (ex: premium bem avaliado, barato com alto desconto, popular problemático)',
  'Qual cluster representa a maior oportunidade de negócio para a Amazon?',
  'Qual é o sentimento predominante nos títulos e conteúdos dos reviews por categoria?',
  'Existe divergência entre o sentimento textual do review e o rating numérico dado?',
  'Quais palavras mais frequentes aparecem em reviews positivos vs. negativos?',
]

answers_by_q = {a['q']: a for a in answers}

lines = []
lines.append('# Amazon Product Intelligence — Business Questions Report')
lines.append('')
lines.append('## Executive KPIs')
lines.append('')
lines.append(f"- Total products: {fmt_int(kpis.get('total_products', 0))}")
lines.append(f"- Average rating: {fmt_float(kpis.get('avg_rating', float('nan')),2)}")
lines.append(f"- Average discount: {fmt_float(kpis.get('avg_discount', float('nan')),2)}%")
lines.append(f"- Top category: {kpis.get('top_category', 'n/a')}")
lines.append(f"- Best PSI: {kpis.get('best_psi_product', 'n/a')} (PSI {fmt_float(kpis.get('best_psi_value', float('nan')),2)})")
lines.append('')
lines.append('## Questions & Answers')
lines.append('')

for i, q in enumerate(expected_q, start=1):
    a = answers_by_q.get(q)
    if a is None:
        continue
    lines.append(f"### {i}. {q}")
    lines.append('')
    lines.append(f"**Answer:** {a['a']}")
    lines.append('')
    lines.append(f"**Insight:** {a['insight']}")
    lines.append('')
    lines.append(f"**Recommendation:** {a['rec']}")
    lines.append('')

report_path = REPORTS_DIR / 'business_questions_report.md'
report_path.write_text("\n".join(lines), encoding='utf-8')
report_path


WindowsPath('C:/Users/flavi/Documents/GitHub/Amazon_Product_Clustering/amazon-product-intelligence/reports/business_questions_report.md')

## Preparar dados embutidos (JSON) para o dashboard

In [6]:
product_cols = [
    'product_id','product_name','main_category','sub_category',
    'discounted_price_clean','actual_price_clean','discount_pct_clean',
    'rating_clean','rating_count_clean','economia_absoluta','faixa_preco','faixa_desconto',
    'PSI','cluster','cluster_name','pca_1','pca_2','img_link','product_link'
]
products_json = df_products[product_cols].replace({np.nan: None}).to_dict(orient='records')

review_cols = [
    'product_id','product_name','main_category','review_title','review_content',
    'rating_clean','sentiment_score','sentimento_label'
]
review_cols = [c for c in review_cols if c in df_reviews.columns]
reviews_json = df_reviews[review_cols].replace({np.nan: None}).to_dict(orient='records')

qa_json = [
    {
        'n': i+1,
        'question': expected_q[i],
        'answer': answers_by_q[expected_q[i]]['a'] if expected_q[i] in answers_by_q else 'n/a',
        'insight': answers_by_q[expected_q[i]]['insight'] if expected_q[i] in answers_by_q else '',
        'recommendation': answers_by_q[expected_q[i]]['rec'] if expected_q[i] in answers_by_q else '',
    }
    for i in range(len(expected_q))
]

dashboard_data = {
    'kpis': kpis,
    'products': products_json,
    'reviews': reviews_json,
    'qa': qa_json,
}
len(products_json), len(reviews_json)


(1351, 1465)

## Gerar dashboard/index.html (standalone offline)

In [7]:
dashboard_path = write_dashboard(dashboard_data, DASHBOARD_DIR / 'index.html')
dashboard_path


WindowsPath('C:/Users/flavi/Documents/GitHub/Amazon_Product_Clustering/amazon-product-intelligence/dashboard/index.html')

## Gerar dashboard_preview.png (imagem representativa)

In [8]:
plt.figure(figsize=(14, 7), facecolor='#0a0e1a')
ax = plt.gca()
ax.set_facecolor('#0a0e1a')
ax.axis('off')
ax.text(0.02, 0.82, 'Amazon Product Intelligence', fontsize=34, color='#e5e7eb', fontweight='bold')
ax.text(0.02, 0.72, 'Pricing, Ratings & Market Segmentation Dashboard', fontsize=16, color='#94a3b8')
ax.text(0.02, 0.52, f"KPIs — Products: {total_products} | Avg Rating: {kpis['avg_rating']} | Avg Discount: {kpis['avg_discount']}% | Clusters: {n_clusters}", fontsize=14, color='#a5b4fc')
ax.add_patch(plt.Rectangle((0.02, 0.18), 0.96, 0.24, color='#111827', alpha=0.9))
ax.text(0.04, 0.34, 'Offline standalone HTML dashboard', fontsize=16, color='#e5e7eb', fontweight='bold')
ax.text(0.04, 0.24, 'Open: dashboard/index.html', fontsize=13, color='#94a3b8')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'dashboard_preview.png', dpi=160, facecolor='#0a0e1a')
plt.close()
FIGURES_DIR / 'dashboard_preview.png'


WindowsPath('C:/Users/flavi/Documents/GitHub/Amazon_Product_Clustering/amazon-product-intelligence/reports/figures/dashboard_preview.png')